# LangGraph + OpenAI Chatbot with Gradio UI

This notebook builds a simple conversational AI chatbot using **LangGraph** for graph-based flow control and **OpenAI GPT-4o-mini** as the language model. The chat interface is served via **Gradio**.

**Flow overview:**
```
User Input → [State] → LangGraph (chatbot node) → OpenAI LLM → Response → Gradio UI
```

In [34]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages   # reducer: merges new messages into existing state
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random


In [35]:
# Load OPENAI_API_KEY and any other secrets from the .env file
load_dotenv(override=True)


True

### Step 1: Define the State object

You can use any python object; but it's most common to use a TypedDict or a Pydantic BaseModel.

`add_messages` is a **reducer** — instead of overwriting the messages list on each graph invocation, it appends new messages to the existing list. This is what gives the graph its memory of previous turns.

In [36]:
class State(BaseModel):
    messages: Annotated[list, add_messages]

### Step 2: Start the Graph Builder with this State class

In [37]:
graph_builder= StateGraph(State)

### Step 3: Create a Node

A node can be any python function.

The reducer that we set before gets automatically called to combine this response with previous responses


In [38]:
llm= ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response= llm.invoke(old_state.messages)      # pass full message history to maintain conversation context
    new_state= State(messages=[response])          # wrap response; add_messages reducer will merge it into history
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

### Step 4: Create Edges

In [39]:
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot", END)

### Step 5: Compile the Graph

In [40]:
graph= graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))  # visualize the graph structure as a diagram

### Step 6: Launch the Gradio Chat Interface

The `chat` function bridges Gradio and LangGraph — it wraps the user input into the `State` schema, runs the graph, and returns only the last AI message for display.

> **Note:** Each call creates a fresh `State` (no cross-turn memory in this version). The Gradio `history` parameter is present for UI compatibility but not passed into the graph.

In [ ]:
def chat(user_input: str, history):
    initial_state= State(messages=[{"role": "user", "content": user_input}])
    result= graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content   # extract the last AI reply from the messages list

gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.


{'messages': [HumanMessage(content='Hi there', additional_kwargs={}, response_metadata={}, id='272e75fd-4026-40a6-94a5-eeb7977b2792'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_888e567758', 'id': 'chatcmpl-DXldZqXVSelCwCwkyF8Vq9DMosvnb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db9e5-6844-7c43-bdc0-be72ed2cc8b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details